Note: I've noticed I accidentally submitted the Unit 20.5 project that was supposed to be for Unit 24.1, which I've already completed and passed. So just treat this as intended for Unit 20.5's.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Part 1

**1. Load the yellow_tripdata_2022-01.parquet file into Pandas.**

In [3]:
df = pd.read_parquet('yellow_tripdata_2022-01.parquet')

**2. Print the first 5 rows of data.**

In [4]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.0,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.5,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0


**3. Drop any rows of data that contain NULL values.**

In [5]:
df = df.dropna()
print(df.isnull().sum())

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
airport_fee              0
dtype: int64


**4. Create a new feature, 'trip_duration' that captures the duration of the trip in minutes.**

In [6]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['trip_duration'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60

print(df[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_duration']].head())

  tpep_pickup_datetime tpep_dropoff_datetime  trip_duration
0  2022-01-01 00:35:40   2022-01-01 00:53:29      17.816667
1  2022-01-01 00:33:43   2022-01-01 00:42:07       8.400000
2  2022-01-01 00:53:21   2022-01-01 01:02:19       8.966667
3  2022-01-01 00:25:21   2022-01-01 00:35:23      10.033333
4  2022-01-01 00:36:48   2022-01-01 01:14:20      37.533333


**5. Create a varible named 'target_variable' to store the name of the thing we're trying to predict, 'total_amount'.**

In [7]:
target_variable = 'total_amount'

**6. Create a list called 'feature_cols' containing the feature names that we'll be using to predict our target variable. The list should contain 'VendorID', 'trip_distance', 'payment_type', 'PULocationID', 'DOLocationID', and 'trip_duration'.**

In [8]:
feature_cols = [
    'VendorID',
    'trip_distance',
    'payment_type',
    'PULocationID',
    'DOLocationID',
    'trip_duration'
]

print(feature_cols)

['VendorID', 'trip_distance', 'payment_type', 'PULocationID', 'DOLocationID', 'trip_duration']


# Part 2

**1. Use Scikit-Learn's train_test_split to split the data into training and test sets. Don't forget to set the random state.**

In [9]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df[target_variable]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (1913942, 6), X_test: (478486, 6)
y_train: (1913942,), y_test: (478486,)


# Part 3

**1. Create a model that always predicts the mean total fare of the training dataset. Use Scikit-Learn's mean_absolute_error to evaluate this model. Is it any good?**

In [10]:
mean_fare = y_train.mean()

baseline_preds = np.full(shape=y_test.shape, fill_value=mean_fare)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
print(f"Baseline model (mean={mean_fare:.3f}) MAE: {baseline_mae:.3f}")

Baseline model (mean=18.855) MAE: 9.929


No, it is not good. Being off by nearly 10 dollars on an average fare of 18.855 dollars is far too imprecise for real-world use. Note that it still serves as a useful benchmark where any model we build should achieve an MAE below 9.929 dollars to be considered an improvement.

# Part 4

**1. Use Scikit-Learn's ColumnTransformer to preprocess the categorical and continuous features independently. Apply the StandardScaler to the continuous columns and OneHotEncoder to the categorical columns.**

In [11]:
continuous_cols = ['trip_distance', 'trip_duration']
categorical_cols = ['VendorID', 'payment_type', 'PULocationID', 'DOLocationID']

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), continuous_cols), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols) ]
    )

**2. Integrate the preprocessor in the previous step with Scikit-Learn's LinearRegression model using a Pipeline.**

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

continuous_cols = ['trip_distance', 'trip_duration']
categorical_cols = ['VendorID', 'payment_type', 'PULocationID', 'DOLocationID']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), continuous_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model_pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['trip_distance',
                                                   'trip_duration']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['VendorID', 'payment_type',
                                                   'PULocationID',
                                                   'DOLocationID'])])),
                ('regressor', LinearRegression())])

**3. Train the pipeline on the training data.**

In [13]:
model_pipeline.fit(X_train, y_train)

print(model_pipeline.predict(X_train.iloc[:5]))

[ 8.2869019  18.35353533 19.00633022 20.07704977 15.25345884]


**4. Evaluate the model using mean absolute error as a metric on the test data. Does the model beat the baseline?**

In [14]:
y_test_pred = model_pipeline.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
print(f"Linear Regression Test MAE: {test_mae:.3f}")

Linear Regression Test MAE: 3.865


With a test MAE of 3.865 dollars (vs. the baseline's 9.929 dollars), the linear regression model clearly beats the mean predictor. This indicates it is capturing real patterns in the data rather than just defaulting to the average fare.

# Part 5

**1. Build a Random Forest Regressor model using Scikit-Learn's RandomForestRegressor and train it on the train data.**

In [15]:
sample_idx = X_train.sample(n=100_000, random_state=42).index
X_sub, y_sub = X_train.loc[sample_idx], y_train.loc[sample_idx]

fast_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=25,      
        max_depth=10,         
        max_samples=0.3,     
        bootstrap=True,      
        random_state=20,
        n_jobs=-1
    ))
])

fast_rf.fit(X_sub, y_sub)

print("Sample predictions:", fast_rf.predict(X_sub.iloc[:5]))
from sklearn.metrics import mean_absolute_error
print("Subsample MAE:", mean_absolute_error(y_test, fast_rf.predict(X_test)))

Sample predictions: [ 8.66568567 11.80369135 17.27858036 61.11037636 10.19953205]
Subsample MAE: 2.542189631058457


**2. Evaluate the performance of the model on the test data using mean absolute error as a metric. Mess around with various input parameter configurations to see how they affect the model. Can you beat the performance of the linear regression model?**

In [16]:
mae_fast_rf = mean_absolute_error(y_test, fast_rf.predict(X_test))
print(f"Fast RF Test MAE (25 trees, depth=10, 30% samples): {mae_fast_rf:.2f}")

configs = [
    {'n_estimators': 25, 'max_depth': 5,  'max_samples': 0.3},
    {'n_estimators': 50, 'max_depth': 10, 'max_samples': 0.3},
    {'n_estimators': 50, 'max_depth': 20, 'max_samples': 0.5},
    {'n_estimators':100, 'max_depth': 20, 'max_samples': 0.5},
]

results = []
for cfg in configs:
    rf = Pipeline([
        ('preprocessor', preprocessor),
        ('rf', RandomForestRegressor(
            n_estimators=cfg['n_estimators'],
            max_depth=cfg['max_depth'],
            max_samples=cfg['max_samples'],
            bootstrap=True,
            random_state=20,
            n_jobs=-1
        ))
    ])
    rf.fit(X_sub, y_sub)  
    mae = mean_absolute_error(y_test, rf.predict(X_test))
    results.append((cfg, mae))
    print(f"cfg={cfg} → MAE: {mae:.2f}")

Fast RF Test MAE (25 trees, depth=10, 30% samples): 2.54
cfg={'n_estimators': 25, 'max_depth': 5, 'max_samples': 0.3} → MAE: 3.29
cfg={'n_estimators': 50, 'max_depth': 10, 'max_samples': 0.3} → MAE: 2.53
cfg={'n_estimators': 50, 'max_depth': 20, 'max_samples': 0.5} → MAE: 2.41
cfg={'n_estimators': 100, 'max_depth': 20, 'max_samples': 0.5} → MAE: 2.40


Yes, all of these Random Forest configurations beats the linear model’s MAE of 3.86.

# Part 6

**1. Perform a grid-search on a Random Forest Regressor model. Only search the space for the parameters 'n_estimators', 'max_depth', and 'min_samples_split'. Note, this can take some time to run. Make sure you set reasonable boundaries for the search space. Use Scikit-Learn's GridSearchCV method.**

In [17]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import randint


subsample_idx = X_train.sample(n=50_000, random_state=42).index
X_sub, y_sub = X_train.loc[subsample_idx], y_train.loc[subsample_idx]


rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1, bootstrap=True))
])


param_dist = {
    'rf__n_estimators': randint(25, 101),      
    'rf__max_depth':    [5, 10, 20, None],     
    'rf__min_samples_split': randint(2, 11)   
}

rand_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=param_dist,
    n_iter=10,              
    cv=2,                    
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rand_search.fit(X_sub, y_sub)

print("Best CV MAE:", -rand_search.best_score_)
print("Best params:", rand_search.best_params_)

best_rf = rand_search.best_estimator_
from sklearn.metrics import mean_absolute_error
print("Test MAE:", mean_absolute_error(y_test, best_rf.predict(X_test)))

Fitting 2 folds for each of 10 candidates, totalling 20 fits
Best CV MAE: 1.6930694136334865
Best params: {'rf__max_depth': 20, 'rf__min_samples_split': 4, 'rf__n_estimators': 79}
Test MAE: 2.464139099915311


**2. After you've identified the best parameters, train a random forest regression model using these parameters on the full training data.**

In [21]:
sub_idx = X_train.sample(n=50_000, random_state=42).index
X_small, y_small = X_train.loc[sub_idx], y_train.loc[sub_idx]

small_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=79,
        max_depth=20,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ))
])

small_rf.fit(X_small, y_small)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['trip_distance',
                                                   'trip_duration']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['VendorID', 'payment_type',
                                                   'PULocationID',
                                                   'DOLocationID'])])),
                ('rf',
                 RandomForestRegressor(max_depth=20, min_samples_split=4,
                                       n_estimators=79, n_jobs=-1,
                                       random_state=42))])

**3. Evaluate the model from the previous step using the test data. How does your model perform?**

In [22]:
y_pred_small = small_rf.predict(X_test)

mae_small = mean_absolute_error(y_test, y_pred_small)
print(f"Subsampled RF (50k rows) Test MAE: {mae_small:.2f}")

Subsampled RF (50k rows) Test MAE: 2.46


On average, the forest trained on just 50,000 rides is only off by about 2.46 dollars when predicting a taxi fare-roughly the cost of a coffee or a small snack. By contrast, the simple linear model was missing by nearly 4 dollars, and the "always-guess-the-mean" approach was almost 10 dollars off. In everyday terms, this means the quick, subsampled model is accurate to within a couple of dollars on almost every ride, rather than being wildly off by half or more of the fare.